In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

import joblib

In [7]:
# Load the dataset
df = pd.read_csv("data/processed/train.csv")
df.head()

,Gender,AttendanceRate,StudyHoursPerWeek,PreviousGrade,ExtracurricularActivities,ParentalSupport,Online Classes Taken,Study_Attendance_Score,Study_per_Activity,FinalGrade
0,Male,91.000000,22.0,77.0,3.0,Medium,False,2002.000000,5.50,62.0
1,Male,70.000000,8.0,78.0,3.0,Low,True,560.000000,2.00,62.0
2,Female,91.000000,15.0,70.0,3.0,Medium,False,1365.000000,3.75,78.0
3,Male,91.000000,22.0,60.0,0.0,Low,False,2002.000000,22.00,87.0
4,Male,85.510417,20.0,70.0,0.0,High,True,1710.208333,20.00,88.0


In [53]:
# Feature tương tác (interaction)
df['Study_Attendance'] = df['StudyHoursPerWeek'] * df['AttendanceRate']

# Feature tỉ lệ
df['Study_per_Attendance'] = df['StudyHoursPerWeek'] / (df['AttendanceRate'] + 1e-5)

# Feature bình phương (phi tuyến)
df['StudyHours_sq'] = df['StudyHoursPerWeek'] ** 2
df['Attendance_sq'] = df['AttendanceRate'] ** 2


In [54]:
# Tách tập dữ liệu thành:
# - X (features): bao gồm các biến đầu vào như thời gian học, điểm trước đó,...
# - y (target): biến mục tiêu FinalGrade cần dự đoán trong bài toán hồi quy

X = df.drop("FinalGrade", axis=1)
y = df["FinalGrade"]

In [55]:
# Phân loại các cột trong dữ liệu:
# - num_cols: các cột dạng số (dùng để scale)
# - cat_cols: các cột dạng phân loại (dùng để encode)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()

print("Numerical:", num_cols)
print("Categorical:", cat_cols)


Numerical: ['AttendanceRate', 'StudyHoursPerWeek', 'PreviousGrade', 'ExtracurricularActivities', 'Study_Attendance_Score', 'Study_per_Activity', 'Study_Attendance', 'Study_per_Attendance', 'StudyHours_sq', 'Attendance_sq']
Categorical: ['Gender', 'ParentalSupport']


In [56]:
# Xây dựng bộ tiền xử lý dữ liệu:
# - Cột số (num_cols): chuẩn hóa bằng StandardScaler
# - Cột phân loại (cat_cols): mã hóa bằng OneHotEncoder
# ColumnTransformer giúp áp dụng các phương pháp khác nhau cho từng nhóm cột

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])


In [57]:
# Xây dựng các Pipeline cho nhiều mô hình học máy khác nhau
# Mỗi pipeline bao gồm:
# - Bước tiền xử lý dữ liệu (preprocessing)
# - Bước huấn luyện mô hình (model)
# Mục tiêu: so sánh hiệu quả của các thuật toán khác nhau

# Linear Regression
from sklearn.linear_model import LinearRegression

pipe_lr = Pipeline([
    ('preprocessing', preprocessor),
    ('model', LinearRegression())
])

In [58]:
# Random Forest
pipe_rf = Pipeline([
    ('preprocessing', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=300,
        max_depth=10,
        min_samples_split=5,
        random_state=42
    ))
])


In [59]:
# SVM
from sklearn.svm import SVR

pipe_svm = Pipeline([
    ('preprocessing', preprocessor),
    ('model', SVR())
])


In [60]:
# Decision Tree Regressor
from sklearn.tree import DecisionTreeRegressor

pipe_dt = Pipeline([
    ('preprocessing', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42))
])


In [61]:
# K-Nearest Neighbors Regressor
from sklearn.neighbors import KNeighborsRegressor

pipe_knn = Pipeline([
    ('preprocessing', preprocessor),
    ('model', KNeighborsRegressor(n_neighbors=5))
])


In [62]:
from sklearn.ensemble import GradientBoostingRegressor

# Gradient Boosting
pipe_gb = Pipeline([
    ('preprocessing', preprocessor),
    ('model', GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=3,
        random_state=42
    ))
])


In [63]:
# Ridge Regression
from sklearn.linear_model import Ridge

pipe_ridge = Pipeline([
    ('preprocessing', preprocessor),
    ('model', Ridge(alpha=1.0))
])


In [64]:
# Xây dựng thước đo RMSE (Root Mean Squared Error) cho bài toán hồi quy
# RMSE đo lường sai số trung bình giữa giá trị dự đoán và thực tế
# Giá trị RMSE càng thấp thì mô hình càng tốt
from sklearn.metrics import make_scorer, mean_squared_error
import numpy as np

rmse_scorer = make_scorer(
    lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
    greater_is_better=False
)


In [65]:
# Đánh giá hiệu năng của các mô hình bằng phương pháp Cross Validation
# Dữ liệu được chia thành nhiều phần (fold), mô hình sẽ train và test nhiều lần
# Mục tiêu: đánh giá độ chính xác một cách ổn định và tránh overfitting
models = {
    "Linear Regression": pipe_lr,
    "Ridge": pipe_ridge,
    "Random Forest": pipe_rf,
    "Decision Tree": pipe_dt,
    "KNN": pipe_knn,
    "Gradient Boosting": pipe_gb,
    "SVM": pipe_svm
}
results = {}

for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=5, scoring='r2')
    results[name] = scores.mean()
    print(f"{name}: {scores.mean():.4f}")


Linear Regression: -0.0144
Ridge: -0.0123
Random Forest: -0.0484
Decision Tree: -1.0228
KNN: -0.1639
Gradient Boosting: -0.0836
SVM: -0.0291


In [66]:
# So sánh hiệu năng của các mô hình dựa trên điểm số R2
# Mô hình có R2 cao hơn sẽ dự đoán tốt hơn
pd.DataFrame.from_dict(results, orient='index', columns=['R2 Score']).sort_values(by='R2 Score', ascending=False)


,R2 Score
Ridge,-0.012313
Linear Regression,-0.014387
SVM,-0.029126
Random Forest,-0.048445
Gradient Boosting,-0.083554
KNN,-0.163888
Decision Tree,-1.022814


Mặc dù Ridge Regression cho kết quả tốt nhất trong các mô hình, tuy nhiên giá trị R2 âm cho thấy mô hình chưa dự đoán tốt. Điều này cho thấy cần cải thiện dữ liệu hoặc feature engineering để nâng cao hiệu quả mô hình.

In [67]:
# Chọn mô hình có kết quả tốt nhất dựa trên R2 score
# Sau đó huấn luyện lại mô hình này trên toàn bộ dữ liệu
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("Best model:", best_model_name)

best_model.fit(X, y)


Best model: Ridge


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessing', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c